# 根据历史评分推荐电影

深度学习期末小组作业

[项目仓库](https://github.com/xunan23333-bit/movielens-sequence-rec)

## 按人切分评分记录

每人留出最后五条测试，紧邻的五条作验证。

In [1]:
from pathlib import Path
import time, copy, json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score

OUT = Path("out")
OUT.mkdir(exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv").set_index("movieId")
user_ids = np.sort(ratings.userId.unique())
movie_ids = np.sort(ratings.movieId.unique())
ratings["u"] = np.searchsorted(user_ids, ratings.userId)
ratings["i"] = np.searchsorted(movie_ids, ratings.movieId) + 1
ratings = ratings.sort_values(["u", "timestamp"], kind="mergesort")
rank = ratings.groupby("u").cumcount(ascending=False)
train = ratings[rank >= 10]
valid = ratings[(rank >= 5) & (rank < 10)]
test = ratings[rank < 5]
U, I = len(user_ids), len(movie_ids) + 1
seen = np.zeros((U, I), dtype=bool)
seen[ratings.u, ratings.i] = True
train_seen = np.zeros_like(seen)
train_seen[train.u, train.i] = True

def histories(frame):
    return [g[["i", "rating", "timestamp"]].to_numpy() for _, g in frame.groupby("u")]

original = histories(train)
assert len(train) + len(valid) + len(test) == len(ratings)
assert (train.groupby("u").timestamp.max() <= valid.groupby("u").timestamp.min()).all()
assert (valid.groupby("u").timestamp.max() <= test.groupby("u").timestamp.min()).all()
print(dev, torch.__version__, len(train), len(valid), len(test))

cuda 2.12.0+cu126 94736 3050 3050


## 历史与候选

评分和电影对齐。时间权重对同秒记录取相同值。固定 105 候选沿用此前的抽样顺序。

In [2]:
def inputs(items, width, decay):
    x = np.zeros((len(items), width), dtype=np.int64)
    r = np.full_like(x, 10)
    dates = np.zeros_like(x)
    for row, history in enumerate(items):
        h = history[-width:]
        x[row, -len(h):] = h[:, 0]
        r[row, -len(h):] = np.rint(h[:, 1] * 2).astype(int) - 1
        dates[row, -len(h):] = h[:, 2]
    age = np.zeros_like(dates)
    for col in range(width - 2, -1, -1):
        age[:, col] = np.where(dates[:, col] == dates[:, col + 1],
                               age[:, col + 1], width - col - 1)
    weights = ((decay ** age) * (x != 0)).astype(np.float32)
    return tuple(torch.as_tensor(a, device=dev) for a in (x, r, weights))

def training_data(sequences, width, decay):
    prefix, users, target = [], [], []
    known = train_seen.copy()
    for u, h in enumerate(sequences):
        known[u, h[:, 0].astype(int)] = True
        for end in range(1, len(h)):
            prefix.append(h[max(0, end-width):end])
            users.append(u)
            target.append(int(h[end, 0]))
    rows = np.array(users)[:, None]
    rng = np.random.default_rng(612)
    negatives = rng.integers(1, I, size=(len(target), 100))
    bad = known[rows, negatives]
    while bad.any():
        negatives[bad] = rng.integers(1, I, size=bad.sum())
        bad = known[rows, negatives]
    assert not known[rows, negatives].any()
    return inputs(prefix, width, decay), torch.as_tensor(np.column_stack((target, negatives)), device=dev)

rng = np.random.default_rng(42)
# Advance past the original baseline draws to preserve the fixed evaluation lists.
for u, h in enumerate(original):
    pool = np.flatnonzero(~train_seen[u, 1:]) + 1
    for _ in h[1:]:
        rng.choice(pool, 1, replace=False)

def candidates(frame):
    output = []
    for u, g in frame.groupby("u"):
        positive = g.i.to_numpy()
        negative = rng.choice(np.flatnonzero(~seen[u, 1:]) + 1, 100, replace=False)
        row = np.r_[positive, negative]
        assert len(positive) == 5 and len(np.unique(row)) == 105
        output.append(row - 1)
    return np.array(output)

test_items = candidates(test)
valid_items = candidates(valid)
order = np.random.default_rng(58123).permutation(U)
select, review = np.sort(order[:305]), np.sort(order[305:])

## 电影特征

电影题材是多标签。均分和次数只用训练集，并用总体均分平滑。

In [3]:
genre = movies.loc[movie_ids, "genres"].str.get_dummies(sep="|").to_numpy()
genre = np.pad(genre, ((1, 0), (0, 0)))
count = np.bincount(train.i, minlength=I)
total = np.bincount(train.i, weights=train.rating, minlength=I)
mean = train.rating.mean()
stats = np.column_stack((((total + 10 * mean) / (count + 10) - mean) / 2,
                         np.log1p(count) / np.log1p(count.max())))
side_tables = {"none": None, "genres": genre, "stats": stats,
               "both": np.column_stack((genre, stats))}

## GRU 与单层查询注意力

两者共用嵌入和点积。GRU 取最后的状态，注意力用查询向量对历史分配权重。

In [4]:
class SequenceModel(nn.Module):
    def __init__(self, kind, width, side):
        super().__init__()
        self.movie = nn.Embedding(I, 64, padding_idx=0)
        self.rating = nn.Embedding(11, 64, padding_idx=10)
        self.register_buffer("side", None if side is None else torch.tensor(side, dtype=torch.float32))
        self.project = None if side is None else nn.Linear(side.shape[1], 64, bias=False)
        self.kind = kind
        if kind == "GRU":
            self.gru = nn.GRU(64, 64, batch_first=True)
        else:
            self.position = nn.Embedding(width + 1, 64)
            self.query = nn.Parameter(torch.zeros(1, 1, 64))
            self.attention = nn.MultiheadAttention(64, 2, batch_first=True, dropout=.15)
            self.norm1 = nn.LayerNorm(64)
            self.norm2 = nn.LayerNorm(64)
            self.ff = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64))
        self.drop = nn.Dropout(.15)
        nn.init.normal_(self.movie.weight, std=.12)
        nn.init.zeros_(self.movie.weight[:1])
        nn.init.normal_(self.rating.weight, std=.03)
        nn.init.zeros_(self.rating.weight[10:])

    def items(self, ids):
        vector = self.movie(ids)
        if self.project is not None:
            vector = vector + self.project(self.side[ids])
        return vector

    def encode(self, x, r, weights):
        history = (self.items(x) + self.rating(r)) * weights.unsqueeze(-1)
        if self.kind == "GRU":
            states, _ = self.gru(history)
            return self.drop(states[:, -1])
        positions = self.position(torch.arange(x.size(1) + 1, device=x.device))
        query = (self.query + positions[-1]).expand(x.size(0), -1, -1)
        source = torch.cat((history + positions[:-1], query), dim=1)
        padding = F.pad(x == 0, (0, 1), value=False)
        update, _ = self.attention(query, source, source, key_padding_mask=padding, need_weights=False)
        state = self.norm1(query + self.drop(update))
        return self.norm2(state + self.drop(self.ff(state)))[:, 0]

    def forward(self, x, r, w, choices):
        return (self.encode(x, r, w)[:, None] * self.items(choices)).sum(-1)

## 同一套训练和评估

用交叉熵给一真一百负打分。每四轮根据全量验证 NDCG@10 保存最佳参数。

In [5]:
discount = 1 / np.log2(np.arange(2, 12))
ideal = discount[:5].sum()

@torch.no_grad()
def score_all(model, history):
    model.eval()
    return (model.encode(*history) @ model.items(torch.arange(1, I, device=dev)).T).cpu().numpy()

def evaluate(scores, candidate, users):
    rows = np.arange(len(users))[:, None]
    ids = candidate[users]
    positive = ids[:, :5]
    values = scores[users[:, None], ids]
    labels = np.broadcast_to(np.r_[np.ones(5), np.zeros(100)], values.shape)
    top = np.argsort(-values, axis=1, kind="stable")[:, :10]
    hits = labels[rows, top]
    ap = (hits * hits.cumsum(1) / np.arange(1, 11)).sum(1) / 5
    ndcg = hits @ discount / ideal
    full = scores[users].copy()
    mask = seen[users, 1:].copy()
    mask[rows, positive] = False
    full[mask] = -np.inf
    top = np.argsort(-full, axis=1, kind="stable")[:, :10]
    full_ndcg = (top[:, :, None] == positive[:, None, :]).any(2) @ discount / ideal
    return {"AUC": float(roc_auc_score(labels.ravel(), values.ravel())), "MAP10": float(ap.mean()),
            "NDCG10": float(ndcg.mean()), "FullNDCG10": float(full_ndcg.mean())}, ndcg

# A perfect top five gives NDCG@10 = 1.
assert np.isclose(discount[:5].sum() / ideal, 1)
 
def fit(kind, sequences, config, seed):
    width, decay, feature = config
    x, pairs = training_data(sequences, width, decay)
    history = inputs(original, width, decay)
    torch.manual_seed(seed)
    model = SequenceModel(kind, width, side_tables[feature]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=.002, weight_decay=1e-4)
    best, best_epoch = -1, 0
    start = time.perf_counter()
    for epoch in range(1, 25):
        model.train()
        for batch in torch.randperm(len(pairs), device=dev).split(768):
            opt.zero_grad(set_to_none=True)
            logits = model(x[0][batch], x[1][batch], x[2][batch], pairs[batch])
            target = torch.zeros(len(batch), dtype=torch.long, device=dev)
            loss = F.cross_entropy(logits, target)
            loss.backward()
            opt.step()
        if epoch % 4 == 0:
            metrics, _ = evaluate(score_all(model, history), valid_items, select)
            if metrics["FullNDCG10"] > best:
                best, best_epoch = metrics["FullNDCG10"], epoch
                weights = copy.deepcopy(model.state_dict())
    model.load_state_dict(weights)
    scores = score_all(model, history)
    chosen, _ = evaluate(scores, valid_items, select)
    audited, users = evaluate(scores, valid_items, review)
    result = dict(kind=kind, config=config, seed=seed, epoch=best_epoch,
                  seconds=round(time.perf_counter()-start, 1),
                  params=sum(p.numel() for p in model.parameters()),
                  selection=chosen, review=audited)
    print(kind, config, seed, best_epoch, result["seconds"], audited, flush=True)
    return model, result, users

## 按验证结果逐步选配置

先看历史长度，再加衰减和侧特征。305 位验证用户用于选配置，其余 305 位复核。先锁定配置，再看测试。

In [6]:
baseline = (40, 1., "none")
checks = [baseline, (20, 1., "none")]
runs, trained = [], {}

def record(key, kind, config, seed, data=original):
    model, row, scores = fit(kind, data, config, seed)
    row["arm"] = key
    runs.append(row)
    (OUT / "progress.json").write_text(json.dumps(runs, indent=2), encoding="utf8")
    return model, scores

def best_config(options):
    return max(options, key=lambda c: next(r["selection"]["FullNDCG10"] for r in runs
               if tuple(r["config"]) == c and r["kind"] == "Query" and r["seed"] == 42))

for config in checks:
    trained[config] = record("main", "Query", config, 42)
width = best_config(checks)[0]
weighted = (width, .95, "none")
trained[weighted] = record("main", "Query", weighted, 42)
history_config = best_config(checks + [weighted])
feature_checks = [history_config]
for feature in ("genres", "stats", "both"):
    config = (*history_config[:2], feature)
    trained[config] = record("main", "Query", config, 42)
    feature_checks.append(config)
winner = best_config(feature_checks)
query2, original2 = record("main", "Query", winner, 7)
if winner == baseline:
    baseline2 = query2
else:
    baseline2, _ = record("main", "Query", baseline, 7)
gru, _ = record("main", "GRU", baseline, 42)
gru2, _ = record("main", "GRU", baseline, 7)
print("Locked config:", winner)

Query (40, 1.0, 'none') 42 20 19.1 {'AUC': 0.9185239236764311, 'MAP10': 0.5684353369763205, 'NDCG10': 0.6875451452258092, 'FullNDCG10': 0.09437827867074321}


Query (20, 1.0, 'none') 42 20 18.6 {'AUC': 0.916514592851384, 'MAP10': 0.5639318240957586, 'NDCG10': 0.6832830379659088, 'FullNDCG10': 0.0800311470674008}


Query (40, 0.95, 'none') 42 20 18.9 {'AUC': 0.9150242192958882, 'MAP10': 0.5578027582617747, 'NDCG10': 0.6799981592248623, 'FullNDCG10': 0.08839501083740657}


Query (40, 0.95, 'genres') 42 20 20.8 {'AUC': 0.92439303413061, 'MAP10': 0.6006528753577934, 'NDCG10': 0.722641258802885, 'FullNDCG10': 0.10732387485022069}


Query (40, 0.95, 'stats') 42 16 19.0 {'AUC': 0.8472353023380813, 'MAP10': 0.5481004423627374, 'NDCG10': 0.6715077971111487, 'FullNDCG10': 0.07438529502432269}


Query (40, 0.95, 'both') 42 20 19.1 {'AUC': 0.9003824778285408, 'MAP10': 0.5939539422326308, 'NDCG10': 0.7092763291443754, 'FullNDCG10': 0.09914430240802855}


Query (40, 0.95, 'genres') 7 24 19.6 {'AUC': 0.9238145014780974, 'MAP10': 0.5997798594847775, 'NDCG10': 0.7167013256570279, 'FullNDCG10': 0.10501043310827492}


Query (40, 1.0, 'none') 7 20 17.8 {'AUC': 0.9186097823165816, 'MAP10': 0.5751704397606037, 'NDCG10': 0.694642829627899, 'FullNDCG10': 0.07802603323492352}


GRU (40, 1.0, 'none') 42 24 14.5 {'AUC': 0.9121821660843858, 'MAP10': 0.5334431433775695, 'NDCG10': 0.6543539172858374, 'FullNDCG10': 0.08786743968234165}


GRU (40, 1.0, 'none') 7 20 14.5 {'AUC': 0.90719551733405, 'MAP10': 0.5370023419203747, 'NDCG10': 0.6563831882328985, 'FullNDCG10': 0.06907782016268268}


Locked config: (40, 0.95, 'genres')


## GAN 增强

只给训练历史短于 60 条的用户补记录。热门对照与 GAN 匹配每人的补入数量、评分和位置。

In [7]:
popular = np.argsort(-count[1:], kind="stable")[:200] + 1
column = {item: j for j, item in enumerate(popular)}
profiles = np.zeros((U, len(popular)), dtype=np.float32)
for row in train.itertuples():
    if row.i in column:
        profiles[row.u, column[row.i]] = row.rating / 5
torch.manual_seed(42)
real = torch.tensor(profiles, device=dev)
generator = nn.Sequential(nn.Linear(32, 128), nn.LeakyReLU(.2),
                          nn.Linear(128, 200), nn.Sigmoid()).to(dev)
critic = nn.Sequential(nn.Linear(200, 128), nn.LeakyReLU(.2), nn.Linear(128, 1)).to(dev)
g_opt = torch.optim.Adam(generator.parameters(), lr=.0002, betas=(.5, .999))
d_opt = torch.optim.Adam(critic.parameters(), lr=.0002, betas=(.5, .999))
bce = nn.BCEWithLogitsLoss()
for epoch in range(400):
    for batch in torch.randperm(U, device=dev).split(64):
        z = torch.randn(len(batch), 32, device=dev)
        yes = torch.ones((len(batch), 1), device=dev)
        d_opt.zero_grad()
        d_loss = bce(critic(real[batch]), yes) + bce(critic(generator(z).detach()), 1 - yes)
        d_loss.backward()
        d_opt.step()
        g_opt.zero_grad()
        g_loss = bce(critic(generator(z)), yes)
        g_loss.backward()
        g_opt.step()

def augment():
    rng = np.random.default_rng(1121)
    gan, hot = [], []
    with torch.no_grad():
        fake = (generator(torch.randn(U, 5, 32, device=dev)) * 5).cpu().numpy()
    for u, h in enumerate(original):
        available = [j for j in range(200) if not train_seen[u, popular[j]]]
        selected = []
        for j in available:
            rating = float(fake[u, :, j].max())
            if rating >= 4:
                selected.append((popular[j], rating))
        rng.shuffle(selected)
        selected = selected[:max(60-len(h), 0)]
        other = rng.choice(popular[available], len(selected), replace=False)
        dates = np.arange(1, len(selected)+1) + h[-1, 2]
        scores = [r for _, r in selected]
        gan.append(np.vstack((h, np.column_stack(([i for i, _ in selected], scores, dates)))))
        hot.append(np.vstack((h, np.column_stack((other, scores, dates)))))
    return gan, hot

gan_set, hot_set = augment()
added = sum(len(a)-len(b) for a, b in zip(gan_set, original))
assert all(len(a) == len(b) and np.array_equal(a[:, 1:], b[:, 1:]) for a, b in zip(gan_set, hot_set))
short = np.array([len(original[u]) < 60 for u in review])
groups = {"original": [trained[winner][1], original2], "gan": [], "popular": []}
for seed in (42, 7):
    for name, data in (("gan", gan_set), ("popular", hot_set)):
        _, scores = record(name, "Query", winner, seed, data)
        groups[name].append(scores)
gan_result = {}
for name, values in groups.items():
    averages = np.mean(values, axis=0)
    gan_result[name] = [float(averages[short].mean()), float(averages[~short].mean())]
delta = np.mean(groups["gan"], axis=0)[short] - np.mean(groups["original"], axis=0)[short]
bootstrap = np.random.default_rng(1729).choice(delta, size=(2000, len(delta)))
interval = np.quantile(bootstrap.mean(1), [.025, .975]).tolist()
print("GAN", added, gan_result, interval)

Query (40, 0.95, 'genres') 42 24 21.9 {'AUC': 0.9189112174146735, 'MAP10': 0.589448607858444, 'NDCG10': 0.7102102865163732, 'FullNDCG10': 0.10686068320935027}


Query (40, 0.95, 'genres') 42 24 20.7 {'AUC': 0.9183026713249127, 'MAP10': 0.5859531615925059, 'NDCG10': 0.7062865402117848, 'FullNDCG10': 0.09807665510378402}


Query (40, 0.95, 'genres') 7 8 22.2 {'AUC': 0.9252441171728031, 'MAP10': 0.5857514962269061, 'NDCG10': 0.7088664255228696, 'FullNDCG10': 0.08409464694581274}


Query (40, 0.95, 'genres') 7 24 20.4 {'AUC': 0.919280268744961, 'MAP10': 0.6086661462399168, 'NDCG10': 0.7266952445690085, 'FullNDCG10': 0.10240577986699403}


GAN 8650 {'original': [0.7752009602764884, 0.6575860383723753], 'gan': [0.7680202292514605, 0.6441523727534683], 'popular': [0.7736471452549798, 0.6525870263404113]} [-0.021335339673636003, 0.00686837774508599]


## 最终测试与实际推荐

主表为两种子的均值。实际推荐用选定配置的种子 42，过滤用户评过的所有电影。

In [8]:
chosen_model = trained[winner][0]
groups_test = {"GRU": (baseline, [gru, gru2]),
               "QueryBase": (baseline, [trained[baseline][0], baseline2]),
               "QuerySelected": (winner, [chosen_model, query2])}
tests = {}
for name, (config, models) in groups_test.items():
    h = inputs(original, *config[:2])
    values = [evaluate(score_all(m, h), test_items, np.arange(U))[0] for m in models]
    tests[name] = dict(seeds=values, mean={k: float(np.mean([v[k] for v in values])) for k in values[0]})
report = dict(winner=winner, runs=runs, test=tests, gan=gan_result,
              added=added, short=int(short.sum()), gan_ci=interval,
              device=torch.cuda.get_device_name() if dev == "cuda" else dev, torch=str(torch.__version__))
(OUT / "results.json").write_text(json.dumps(report, indent=2), encoding="utf8")
print(pd.DataFrame({k: v["mean"] for k, v in tests.items()}).T.round(4))

scores = score_all(chosen_model, inputs(histories(ratings), *winner[:2]))
scores[seen[:, 1:]] = -np.inf
top = np.argsort(-scores, axis=1, kind="stable")[:, :10]
ids = movie_ids[top].ravel()
recommend = pd.DataFrame(dict(userId=np.repeat(user_ids, 10), rank=np.tile(np.arange(1, 11), U),
                              movieId=ids, title=movies.loc[ids, "title"].to_numpy()))
assert recommend.groupby("userId").size().eq(10).all()
assert not recommend.duplicated(["userId", "movieId"]).any()
assert recommend.merge(ratings[["userId", "movieId"]], on=["userId", "movieId"]).empty
recommend.to_csv(OUT / "top10.csv", index=False, encoding="utf-8-sig")
print(len(recommend), "recommendations")
display(recommend[recommend.userId.isin([1, 53])])

                  AUC   MAP10  NDCG10  FullNDCG10
GRU            0.8838  0.4470  0.5744      0.0482
QueryBase      0.8944  0.4783  0.6068      0.0494
QuerySelected  0.8928  0.4988  0.6242      0.0578


6100 recommendations


,userId,rank,movieId,title
0,1,1,2683,Austin Powers: The Spy Who Shagged Me (1999)
1,1,2,2706,American Pie (1999)
2,1,3,785,Kingpin (1996)
3,1,4,3301,"Whole Nine Yards, The (2000)"
4,1,5,1391,Mars Attacks! (1996)
5,1,6,2598,Pushing Tin (1999)
6,1,7,413,Airheads (1994)
7,1,8,3173,Any Given Sunday (1999)
8,1,9,2539,Analyze This (1999)
9,1,10,104,Happy Gilmore (1996)
